In [ ]:
import enum
from enum import Enum
from fractions import Fraction
from typing import NamedTuple

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse
import tqdm.notebook

from gacha_model import COND_PROB_5_STAR, COND_PROB_6_STAR, PityModel
from plot_tools import draw_multi_cdf_fig, draw_multi_pmf_cdf_fig, draw_pmf_cdf_fig
from random_variable import FiniteDist

In [ ]:
# 在联动寻访中获得若干名 UP 6★ 干员所需抽数的分布

type 状态类 = 过渡态类 | 吸收态类


class 过渡态类(NamedTuple):
    六星水位: int
    已获取UP六星干员数量: int


class 吸收态类(Enum):
    pass


def 获取状态(*, 六星水位: int, 已获取UP六星干员数量: int) -> 状态类:
    return 过渡态类(
        六星水位=六星水位,
        已获取UP六星干员数量=min(已获取UP六星干员数量, 已获取UP六星干员数量上限),
    )


def 状态转移(起始状态: 状态类, *, 是第120抽: bool) -> list[tuple[状态类, float]]:
    转移概率列表: list[tuple[状态类, float]] = []

    if isinstance(起始状态, 吸收态类):
        转移概率列表.append((起始状态, 1))

    else:
        起始六星水位 = 起始状态.六星水位
        起始已获取UP六星干员数量 = 起始状态.已获取UP六星干员数量

        if 是第120抽 and 起始已获取UP六星干员数量 == 0:  # 第120抽时，若未获取过UP六星干员，则必定获取
            转移概率列表.append((获取状态(六星水位=0, 已获取UP六星干员数量=1), 1))

        else:
            # 计算抽到不同星级干员的概率
            六星概率 = COND_PROB_6_STAR[起始六星水位]

            # 抽到UP6星干员
            转移概率列表.append((获取状态(六星水位=0, 已获取UP六星干员数量=起始已获取UP六星干员数量 + 1), 六星概率 / 2))

            # 抽到其他6星干员
            转移概率列表.append((获取状态(六星水位=0, 已获取UP六星干员数量=起始已获取UP六星干员数量), 六星概率 / 2))

            # 抽到非6星干员
            转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 已获取UP六星干员数量=起始已获取UP六星干员数量), 1 - 六星概率))

    转移概率列表 = [(目标状态, 概率) for 目标状态, 概率 in 转移概率列表 if 概率 > 0]
    assert np.isclose(sum(x[1] for x in 转移概率列表), 1)
    return 转移概率列表


已获取UP六星干员数量上限 = 6


状态列表: list[状态类] = []
状态列表.extend(过渡态类(六星水位=六星水位, 已获取UP六星干员数量=已获取UP六星干员数量)
            for 六星水位 in range(0, 99, 1)
            for 已获取UP六星干员数量 in range(0, 已获取UP六星干员数量上限 + 1, 1))
状态数量: int = len(状态列表)
状态索引: dict[状态类, int] = {状态: i for i, 状态 in enumerate(状态列表)}

row = []
col = []
data = []
for 起始状态序号, 起始状态 in tqdm.notebook.tqdm(list(enumerate(状态列表))):
    转移概率列表 = 状态转移(起始状态, 是第120抽=False)
    for 目标状态, 概率 in 转移概率列表:
        目标状态序号 = 状态索引[目标状态]
        row.append(起始状态序号)
        col.append(目标状态序号)
        data.append(概率)
状态转移矩阵_非第10抽且非第120抽 = scipy.sparse.csr_matrix((data, (row, col)), shape=(状态数量, 状态数量))

row = []
col = []
data = []
for 起始状态序号, 起始状态 in tqdm.notebook.tqdm(list(enumerate(状态列表))):
    转移概率列表 = 状态转移(起始状态, 是第120抽=True)
    for 目标状态, 概率 in 转移概率列表:
        目标状态序号 = 状态索引[目标状态]
        row.append(起始状态序号)
        col.append(目标状态序号)
        data.append(概率)
状态转移矩阵_第120抽 = scipy.sparse.csr_matrix((data, (row, col)), shape=(状态数量, 状态数量))

In [ ]:
迭代次数 = 4096

初始状态 = 过渡态类(六星水位=0, 已获取UP六星干员数量=0)
当前状态分布 = np.zeros(状态数量)
当前状态分布[状态索引[初始状态]] = 1

历史状态分布 = np.zeros((迭代次数 + 1, 状态数量))
历史状态分布[0] = 当前状态分布

for i in tqdm.notebook.tqdm(range(迭代次数)):
    if i == 119:
        状态转移矩阵 = 状态转移矩阵_第120抽
    else:
        状态转移矩阵 = 状态转移矩阵_非第10抽且非第120抽
    当前状态分布 = 当前状态分布 @ 状态转移矩阵

    历史状态分布[i + 1] = 当前状态分布

In [ ]:
dists = [FiniteDist([1])]

for 已获取UP六星干员数量 in range(1, 已获取UP六星干员数量上限 + 1):
    成功状态序号列表 = [状态索引[状态] for 状态 in 状态列表 if isinstance(状态, 过渡态类) and 状态.已获取UP六星干员数量 >= 已获取UP六星干员数量]
    cdf = np.sum(历史状态分布[:, 成功状态序号列表], axis=1)
    pmf = np.clip(np.diff(cdf, prepend=0), 0, 1)
    dist = FiniteDist(pmf)
    dists.append(dist)

In [ ]:
draw_multi_pmf_cdf_fig(dists=dists[1:],
                       labels=[f"{n} 名" for n in range(1, 已获取UP六星干员数量上限 + 1)],
                       title="在联动寻访中获得若干名 UP 6★ 干员所需抽数的分布",
                       quantile_poses=[0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99],
                       x_max=1024,
                       ax1_y_top=0.025,
                       drawstyle="steps-mid")

In [ ]:
draw_pmf_cdf_fig(dist=dists[1],
                 title="在联动寻访中获得 1 名 UP 6★ 干员所需抽数的分布",
                 quantile_poses=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99],
                 x_max=192,
                 drawstyle="steps-mid")